In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/Users/samantha/QuantUS-Plugins-CEUS/China_Data
/Users/samantha/QuantUS-Plugins-CEUS


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'nifti', 'custom_dicom', 'mp4']


In [137]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/p39/v1/CEUS-29817.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [138]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [139]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [140]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/p39/v1/voi_necrotic_removed.nii.gz'
seg_loader_kwargs = {}

In [141]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis

In [142]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [143]:
analysis_type = 'curves_paramap'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['pyradiomics', 'tic']


In [144]:
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['cor_vox_ovrlp', 'sag_vox_len', 'ax_vox_len', 'ax_vox_ovrlp', 'sag_vox_ovrlp', 'cor_vox_len']


In [145]:
# Set frame rate (adjust this value to match your actual video fps)
image_data.frame_rate = 1  # e.g., 30 fps

analysis_kwargs = {
    'ax_vox_ovrlp': 50,
    'sag_vox_ovrlp': 50,
    'cor_vox_ovrlp': 50,
    'ax_vox_len': 20.0,
    'sag_vox_len': 20.0,
    'cor_vox_len': 20.0,
}


In [146]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves: 100%|██████████| 228/228 [02:26<00:00,  1.55it/s]


## Curve Quantification

In [147]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'dte', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select'])


In [148]:
function_names = [] # Empty list will use all functions
output_path = '/Users/samantha/Desktop/ultrasound lab stuff/china data/output/curve_quant_raw.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [149]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

In [150]:
import numpy as np
data = curve_quant.data_dict[0]
print(f"{'Parameter':<30} {'Value':>15}")
print("-" * 47)
for key, value in data.items():
    if isinstance(value, float):
        print(f"{key:<30} {value:>15.4f}")
    else:
        print(f"{key:<30} {str(value):>15}")

# Volume calculation
pixdim = seg_data.pixdim
voxel_vol_mm3 = np.prod(pixdim)
n_voxels = int(np.sum(seg_data.seg_mask > 0))
vol_mm3 = n_voxels * voxel_vol_mm3

print(f"\n{'VOI Volume':<30}")
print("-" * 47)
print(f"{'Voxel spacing (mm)':<30} {str(pixdim):>15}")
print(f"{'Voxels in VOI':<30} {n_voxels:>15,}")
print(f"{'Volume (mm³)':<30} {vol_mm3:>15.1f}")
print(f"{'Volume (cm³)':<30} {vol_mm3/1000:>15.2f}")

Parameter                                Value
-----------------------------------------------
Scan Name                       CEUS-29817.nii
Segmentation Name              voi_necrotic_removed
Window-Axial Start Pix                      24
Window-Sagittal Start Pix                   52
Window-Axial End Pix                        38
Window-Sagittal End Pix                     66
Window-Coronal Start Pix                    82
Window-Coronal End Pix                      89
AUC_NoFit_TIC                         102.7306
DTE_TIC                               -34.3522
Mean_select_TIC                        29.9246
Std_select_TIC                          8.3150
Max_select_TIC                         66.1894
Min_select_TIC                         18.7317
Median_select_TIC                      27.6114
Variance_select_TIC                    69.1393
Skewness_select_TIC                     2.0692
Kurtosis_select_TIC                     5.1726
Range_select_TIC                       47.4578
Interqu

In [151]:
import numpy as np
data = curve_quant.data_dict[0]
print(f"{'Parameter':<30} {'Value':>15}")
print("-" * 47)
for key, value in data.items():
    if isinstance(value, float):
        print(f"{key:<30} {value:>15.4f}")
    else:
        print(f"{key:<30} {str(value):>15}")

# Volume calculation
pixdim = seg_data.pixdim
voxel_vol_mm3 = np.prod(pixdim)
n_voxels = int(np.sum(seg_data.seg_mask > 0))
vol_mm3 = n_voxels * voxel_vol_mm3

print(f"\n{'VOI Volume':<30}")
print("-" * 47)
print(f"{'Voxel spacing (mm)':<30} {str(pixdim):>15}")
print(f"{'Voxels in VOI':<30} {n_voxels:>15,}")
print(f"{'Volume (mm³)':<30} {vol_mm3:>15.1f}")
print(f"{'Volume (cm³)':<30} {vol_mm3/1000:>15.2f}")

Parameter                                Value
-----------------------------------------------
Scan Name                       CEUS-29817.nii
Segmentation Name              voi_necrotic_removed
Window-Axial Start Pix                      24
Window-Sagittal Start Pix                   52
Window-Axial End Pix                        38
Window-Sagittal End Pix                     66
Window-Coronal Start Pix                    82
Window-Coronal End Pix                      89
AUC_NoFit_TIC                         102.7306
DTE_TIC                               -34.3522
Mean_select_TIC                        29.9246
Std_select_TIC                          8.3150
Max_select_TIC                         66.1894
Min_select_TIC                         18.7317
Median_select_TIC                      27.6114
Variance_select_TIC                    69.1393
Skewness_select_TIC                     2.0692
Kurtosis_select_TIC                     5.1726
Range_select_TIC                       47.4578
Interqu

## Save Results to CSV

In [133]:
import pandas as pd
import os
from datetime import datetime

out_path = "/Users/samantha/Desktop/ultrasound lab stuff/china data/p39/v1/paramap3/curve_quant.csv"

data = curve_quant.data_dict[0]
df = pd.DataFrame([data])
df["Timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

scan_col = "Scan Name"

if os.path.exists(out_path):
    existing = pd.read_csv(out_path)
    if "Timestamp" not in existing.columns:
        existing["Timestamp"] = pd.NaT
        existing.to_csv(out_path, index=False)
    df.to_csv(out_path, mode="a", header=False, index=False)
    print(f"Appended to {out_path}")
else:
    df.to_csv(out_path, index=False)
    print(f"Created {out_path}")

Appended to /Users/samantha/Desktop/ultrasound lab stuff/china data/p39/v1/paramap3/curve_quant.csv
